### We have saved the matrix in npy and csv format. It contains data with phase shift [0], conductivity [1], width parameters between curves [2], and degree of curvature parameter a [3].

What needs to be done to prepare this data for processing?
1. Clearly specify which parameter is incoming and what is the label for it.
    **Assumption** Probably our incoming parameters will be mode_phase and mode_tran. These will be 2 inputs. As a Y (label), we can transmit a matrix with a combined description of the cell geometry (yes, we will do that). So, we extract shape feature [2, 3] and accumulate them into seperated matrix 'shape_parameters'. In total, we get the label Y at the output. Thus, based on the requirements for phase displacement at a specific point in space, we will be able to create a geometry without calculating a new cell and simulating it. It's just that the model will assume the shape of a cell at a specific point in space (on the lens), then use simplified 2D calculations to evaluate the FOM of this lens and refine the results.

2. Divide the generated data set into a test and a train
3. Select the basic hyperparameters and create a simple fully connected network
4. Create a loop with assumption, error calculation, optimization
5. Add a training schedule for the model
6. Save the model

In [ ]:
import numpy as np
import torch
from torch import nn

from sklearn.model_selection import train_test_split

In [ ]:
loaded_data_npy = np.load('vectors.npy')

In [ ]:
type(loaded_data_npy)

In [ ]:
loaded_data_npy.shape

In [ ]:
X_input = loaded_data_npy[:,:1] # mode_phase and mode_tran. Transparency is found in the index [1]. We don't need it for training - we just don't pass these indexes and values to the model.
Y_labels = loaded_data_npy[:,2:] # width and a

In [ ]:
print("X:\n", X_input[:10,:])
print("Y:\n", Y_labels[:10,:])

In [ ]:
# turn numpy into torch
X_input = torch.from_numpy(X_input).type(torch.float)
Y_labels = torch.from_numpy(Y_labels).type(torch.float)
X_input[:10], Y_labels[:10]

In [ ]:
X_input.size()

In [ ]:
X_input.shape

In [ ]:
X_input_train, X_input_test, Y_labels_train, Y_labels_test = train_test_split(X_input, Y_labels, test_size=0.2)

In [ ]:
len(X_input_train), len(X_input_test), len(Y_labels_train), len(Y_labels_test)

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
#1. Let's creat the model using nn.Sequential()
model_shape_prediction = nn.Sequential(
    nn.Linear(in_features=1, out_features=32, bias=False), 
    nn.ReLU(),
    nn.Tanh(),
    nn.ReLU(),
    nn.Tanh(),
    nn.ReLU(),
    nn.Linear(in_features=32, out_features=2)
).to(device)

In [ ]:
model_shape_prediction

In [ ]:
loss_fn = nn.L1Loss()

In [ ]:
optimizer = torch.optim.Adam(
    params=model_shape_prediction.parameters(),
    lr=0.001
)

In [ ]:
import matplotlib.pyplot as plt

epochs = 10000

# Put data to target device
X_input_train, Y_labels_train = X_input_train.to(device), Y_labels_train.to(device)
X_input_test, Y_labels_test = X_input_test.to(device), Y_labels_test.to(device)

# Lists for saving loss values
train_losses = []
test_losses = []

for epoch in range(epochs):
    #### Training
    model_shape_prediction.train()

    # 1. Forward pass
    Y_labels_pred = model_shape_prediction(X_input_train)
    
    # 2. Calculate loss
    loss = loss_fn(Y_labels_pred, Y_labels_train)

    # 3. Optimizer zero grad
    optimizer.zero_grad()

    # 4. Loss backward (backpropagation)
    loss.backward()

    # 5. Optimizer step (gradient descent)
    optimizer.step()

    ### Testing
    model_shape_prediction.eval()
    with torch.inference_mode():
        # 1. Forward pass
        test_pred = model_shape_prediction(X_input_test)
        # 2. Calculate test loss
        test_loss = loss_fn(test_pred, Y_labels_test)

    # Saving loss values
    train_losses.append(loss.item())
    test_losses.append(test_loss.item())

    # Print out what's happening
    if epoch % 500 == 0:
        print(f"Epoch: {epoch} | Loss: {loss:.5f}, Test_loss: {test_loss:.5f}")

# Plotting after completing the training
plt.figure(figsize=(10, 10))
plt.plot(range(epochs), train_losses, label='Train Loss', marker='o', markevery=100)
plt.plot(range(epochs), test_losses, label='Test Loss', marker='s', markevery=100)
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.title('Training and Testing Loss')
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
torch.save(model_shape_prediction, "model_shape_prediction.pth")

In [ ]:
loaded_model = torch.load("model_shape_prediction.pth")

In [ ]:
X_new_data = torch.tensor([-3.0])
X_new_data = X_new_data.to(torch.float32)  # We convert it to the float32 type (if necessary)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
X_new_data = X_new_data.to(device)
loaded_model = loaded_model.to(device)
X_new_data.shape

In [ ]:
X_new_data.ndim

In [ ]:
# Setting the evaluation mode
loaded_model.eval()

In [ ]:
# Forecasting
with torch.no_grad():
    prediction = loaded_model(X_new_data)
    
# Output
print("Model output data:", prediction)

print(prediction.shape)
print(prediction.size())
print(type(prediction[0]))


**Note**

We have implemented filling the entire space of shapes and geometries without gaps using a machine learning model. In the example code on which I am based, this moment is implemented based on interpolation.:

gdc = np.linspace(0.1, 0.9, 30)  # grating duty cycle
mode_tran = np.empty(gdc.size)
mode_phase = np.empty(gdc.size)
for n in range(gdc.size):
    mode_tran[n], mode_phase[n] = grating(gp, gh, [gdc[n]])

gdc_new = np.linspace(0.16, 0.65, 500)
mode_phase_interp = np.interp(gdc_new, gdc, mode_phase)
print(f"phase-range:, {mode_phase_interp.max() - mode_phase_interp.min():.6f}")